# Criação de origem_destino_customedio.csv

Esse notebook é utilizado para carregar os dados brutos e gerar a tabela `dados_processados/origem_destino_customedio.csv`. Essa tabela nos informa o preço médio cobrado, mensalmente, por cada compania aérea, para cada Origem -> Destino.

In [4]:
import glob
import pandas as pd 
import numpy as np  

A base de dados nos dá informações sobre quanto foi pago em tarifas aéreas no ano de 2026. Apesar de estar incompleta, tem dados até setembro, o que já serve bem. Esse código lê as tabelas sobre tarifas aéreas de cada mês disponível em 2026:

In [5]:
# sintaxe de generator function. funciona na base da fé!
df = pd.concat(
    (pd.read_csv(f, sep=';', encoding='latin-1') for f in glob.glob('dados_brutos/*.csv')),
    ignore_index=True,
)
df.sample(5)

,nr_ano_referencia,nr_mes_referencia,sg_empresa_icao,sg_icao_origem,sg_icao_destino,nr_tarifa,nr_assentos
3640198,2026,5,GLO,SBCY,SBNF,"582,25",3
6126368,2026,8,GLO,SBGR,SBFI,"926,14",2
5561194,2026,7,TAM,SBRF,SBGR,"1211,9",9
2951019,2026,4,TAM,SBKP,SBFZ,"1340,01",3
2160819,2026,3,TAM,SBPJ,SBGR,"664,91",1


In [6]:
df.columns

df = df.rename(columns={'nr_ano_referencia': 'ano', 'nr_mes_referencia': 'mes', 'sg_empresa_icao': 'empresa',
                   'sg_icao_origem': 'origem', 'sg_icao_destino': 'destino', 'nr_tarifa': "valor_tarifa"})

Ajustando o tipo da coluna 'valor_tarifa' de str para float

In [7]:
display(df.dtypes)
# O valor da tarifa está como str!
print(df.columns)

def valor_tarifa_para_float(valor: str) -> float:
    valor = valor.replace(",", ".")
    valor = float(valor)
    valor = np.float16(valor)
    return valor

df['valor_tarifa'] = df['valor_tarifa'].apply(valor_tarifa_para_float)

df.dtypes

# Eba!!!

ano             int64
mes             int64
empresa           str
origem            str
destino           str
valor_tarifa      str
nr_assentos     int64
dtype: object

Index(['ano', 'mes', 'empresa', 'origem', 'destino', 'valor_tarifa',
       'nr_assentos'],
      dtype='str')


ano               int64
mes               int64
empresa             str
origem              str
destino             str
valor_tarifa    float16
nr_assentos       int64
dtype: object

Verificando dados faltantes (não há células em branco!)

In [8]:
print(f"Dados faltantes totais:\n{df.isna().sum()}")
# AEEE!!!

Dados faltantes totais:
ano             0
mes             0
empresa         0
origem          0
destino         0
valor_tarifa    0
nr_assentos     0
dtype: int64


In [9]:
df["custo_por_assento"] = df["valor_tarifa"] / df["nr_assentos"]
df.sample(5)

c:\Users\Diogo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pandas\io\formats\format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,ano,mes,empresa,origem,destino,valor_tarifa,nr_assentos,custo_por_assento
1230455,2026,2,TAM,SBCT,SBUL,483.25,2,241.625000
3914036,2026,5,TAM,SBSP,SBMO,1316.00,1,1316.000000
1480320,2026,3,AZU,SBCF,SBSL,748.00,3,249.333333
1872996,2026,3,GLO,SBMQ,SBKP,816.00,1,816.000000
1098340,2026,2,GLO,SBGL,SBJP,722.00,2,361.000000


In [10]:
# reset_index faz ser retornado um df
origem_destino_customedio = df.groupby(["mes", "empresa", "origem", "destino"])["custo_por_assento"].mean().reset_index()
origem_destino_customedio

,mes,empresa,origem,destino,custo_por_assento
0,1,ABJ,SBSV,SDLO,346.126543
1,1,ABJ,SBSV,SNCL,190.978720
2,1,ABJ,SDLO,SBSV,321.578571
3,1,ABJ,SNCL,SBSV,269.143711
4,1,AZU,SBAE,SBAR,498.125000
...,...,...,...,...,...
96122,9,TAM,SBZM,SBSR,541.170455
96123,9,TAM,SBZM,SBSV,541.013141
96124,9,TAM,SBZM,SBTE,771.428571
96125,9,TAM,SBZM,SBUL,624.383333


In [11]:
origens_destinos_unicos = df.groupby(["origem", "destino"])["custo_por_assento"].mean().reset_index()
origens_destinos_unicos.to_csv(path_or_buf="dados_processados/origens_destinos_unicos.csv", sep=";")

In [12]:
origens_destinos_unicos.sample(5)

,origem,destino,custo_por_assento
2671,SBGO,SNCP,863.094444
3825,SBKP,SBIH,3072.386364
1911,SBEG,SBCX,832.710717
4184,SBMG,SBFI,1763.753788
1961,SBEG,SBRD,3351.000000


Gravando o resultado na tabela `origem_destino_customedio.csv`.

In [13]:
origem_destino_customedio.to_csv(path_or_buf="dados_processados/origem_destino_customedio.csv", sep=";")

In [15]:
origem_destino_customedio["empresa"].unique()

<ArrowStringArray>
['ABJ', 'AZU', 'CQB', 'GLO', 'TAM']
Length: 5, dtype: str